# Calibration and Thresholding

A classifier can rank examples well and still make bad probability-based decisions. This notebook bridges that gap by focusing on two closely related questions:

- Are the model's predicted probabilities **trustworthy**?
- Given those probabilities, what decision **threshold** should we actually use?

We'll work through a binary classification example where false negatives are expensive, then use **reliability diagrams**, **Brier score**, **expected calibration error (ECE)**, and **cost-sensitive thresholding** to turn raw model scores into better decisions.
        


## Setup and Configuration

We'll keep every experimental choice in one place so it is easy to reason about the effect of calibration choices and threshold rules.
        


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    fbeta_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CONFIG = {
    "seed": 7,
    "n_samples": 6000,
    "n_features": 20,
    "positive_rate": 0.18,
    "label_noise": 0.05,
    "n_bins": 10,
    "rf_estimators": 250,
    "rf_min_samples_leaf": 3,
    "logistic_c": 1.0,
    "fp_cost": 1.0,
    "fn_cost": 6.0,
}

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)


## Reproducibility First

Calibration is sensitive to small distribution shifts, so we'll fix the random seed before doing anything else.
        


In [ ]:
np.random.seed(CONFIG["seed"])
rng = np.random.default_rng(CONFIG["seed"])

print("Configuration")
for key, value in CONFIG.items():
    print(f"- {key}: {value}")
        


## Create a Classification Problem

We want a setting where probability quality matters. We'll generate an imbalanced binary classification dataset with some label noise so the problem feels more realistic than a perfectly separable toy dataset.
        


In [ ]:
X, y = make_classification(
    n_samples=CONFIG["n_samples"],
    n_features=CONFIG["n_features"],
    n_informative=10,
    n_redundant=4,
    n_repeated=0,
    n_clusters_per_class=2,
    weights=[1 - CONFIG["positive_rate"], CONFIG["positive_rate"]],
    class_sep=0.9,
    flip_y=CONFIG["label_noise"],
    random_state=CONFIG["seed"],
)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=CONFIG["seed"],
)

X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=CONFIG["seed"],
)

summary = pd.DataFrame(
    {
        "split": ["train", "calibration", "test"],
        "samples": [len(y_train), len(y_cal), len(y_test)],
        "positive_rate": [y_train.mean(), y_cal.mean(), y_test.mean()],
    }
)

display(summary)
        


## Check the Class Balance

The positive class is intentionally the minority class. That makes thresholding more interesting because a default threshold of 0.5 is often not aligned with the real operating goal.
        


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
class_counts = pd.Series(y_train).value_counts().sort_index()
ax.bar(["negative (0)", "positive (1)"], class_counts.values, color=["#4C78A8", "#E45756"])
ax.set_title("Training Set Class Balance")
ax.set_ylabel("Number of examples")
for index, value in enumerate(class_counts.values):
    ax.text(index, value + 20, f"{value}", ha="center")
plt.show()
        


## Train Two Models with Different Probability Behavior

We'll compare a **logistic regression** model and a **random forest**. Logistic regression often produces smoother probabilities, while tree ensembles can achieve strong ranking performance but less reliable confidence estimates.
        


In [ ]:
logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(C=CONFIG["logistic_c"], max_iter=2000, random_state=CONFIG["seed"]),
        ),
    ]
)

random_forest = RandomForestClassifier(
    n_estimators=CONFIG["rf_estimators"],
    min_samples_leaf=CONFIG["rf_min_samples_leaf"],
    random_state=CONFIG["seed"],
)

logistic_model.fit(X_train, y_train)
random_forest.fit(X_train, y_train)

raw_model_probabilities = {
    "Logistic regression": logistic_model.predict_proba(X_test)[:, 1],
    "Random forest": random_forest.predict_proba(X_test)[:, 1],
}

print("Finished fitting the base models.")
        


## A First Comparison: Ranking vs Probability Quality

Before plotting anything, let's compare several metrics:

- **ROC-AUC** and **average precision** tell us how well the model ranks examples.
- **Brier score** and **log loss** punish bad probabilities.

A model can look strong on ranking metrics and still be poorly calibrated.
        


In [ ]:
def summarize_probabilities(name, y_true, y_prob):
    clipped_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, clipped_prob),
        "average_precision": average_precision_score(y_true, clipped_prob),
        "brier_score": brier_score_loss(y_true, clipped_prob),
        "log_loss": log_loss(y_true, clipped_prob),
        "mean_probability": clipped_prob.mean(),
    }

baseline_metrics = pd.DataFrame(
    [summarize_probabilities(name, y_test, probs) for name, probs in raw_model_probabilities.items()]
).sort_values("brier_score")

display(baseline_metrics)
        


## Inspect the Probability Distributions

A reliability problem often starts to show up in the distribution of predicted probabilities. Are probabilities concentrated near 0 and 1? Are there large regions where the model is confidently wrong?
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (name, probs) in zip(axes, raw_model_probabilities.items()):
    ax.hist(probs, bins=20, color="#72B7B2", edgecolor="white")
    ax.set_title(name)
    ax.set_xlabel("Predicted probability of class 1")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()
        


## Build Calibration Helpers

To reason about calibration, we'll group predictions into bins and compare two numbers inside each bin:

- the average predicted probability
- the empirical fraction of positives

If those values match, the model is calibrated in that region.
        


In [ ]:
def calibration_table(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    clipped_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(clipped_prob, bins[1:-1], right=True)

    rows = []
    for bin_index in range(n_bins):
        mask = bin_ids == bin_index
        if not np.any(mask):
            continue
        rows.append(
            {
                "bin": bin_index,
                "count": int(mask.sum()),
                "avg_confidence": clipped_prob[mask].mean(),
                "empirical_rate": y_true[mask].mean(),
            }
        )

    table = pd.DataFrame(rows)
    table["gap"] = (table["avg_confidence"] - table["empirical_rate"]).abs()
    table["bin_weight"] = table["count"] / len(y_true)
    return table


def expected_calibration_error(y_true, y_prob, n_bins=10):
    table = calibration_table(y_true, y_prob, n_bins=n_bins)
    return float((table["gap"] * table["bin_weight"]).sum())


def plot_reliability_diagram(ax, y_true, y_prob, label, color, n_bins=10):
    table = calibration_table(y_true, y_prob, n_bins=n_bins)
    ax.plot(table["avg_confidence"], table["empirical_rate"], marker="o", linewidth=2, label=label, color=color)
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6, label="perfect calibration")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed positive rate")
    ax.set_title("Reliability Diagram")
        


## Reliability Diagrams for the Raw Models

Points above the diagonal mean the model was **under-confident** in that region. Points below the diagonal mean the model was **over-confident**.
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
colors = ["#4C78A8", "#F58518"]

for ax, (name, probs), color in zip(axes, raw_model_probabilities.items(), colors):
    plot_reliability_diagram(ax, y_test, probs, label=name, color=color, n_bins=CONFIG["n_bins"])
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()
        


## Quantify Calibration Error Explicitly

The diagram is visual, but we also want a compact calibration summary. We'll add **ECE** to the earlier metric table.
        


In [ ]:
calibration_summary = baseline_metrics.copy()
calibration_summary["ece"] = calibration_summary["model"].map(
    {
        name: expected_calibration_error(y_test, probs, n_bins=CONFIG["n_bins"])
        for name, probs in raw_model_probabilities.items()
    }
)

display(calibration_summary.sort_values(["ece", "brier_score"]))
        


## Look Inside the Random Forest Bins

Tree ensembles often produce probabilities that bunch together in a few regions. A bin-level table makes that behavior concrete.
        


In [ ]:
rf_bin_table = calibration_table(y_test, raw_model_probabilities["Random forest"], n_bins=CONFIG["n_bins"])
display(rf_bin_table)
        


## Calibrate the Random Forest

Now we'll keep the random forest's ranking behavior but adjust its probabilities using a separate calibration split.

We'll try two standard approaches:

- **Sigmoid (Platt scaling)**: learns a smooth logistic mapping
- **Isotonic regression**: learns a more flexible monotonic mapping
        


In [ ]:
rf_for_sigmoid = clone(random_forest)
rf_for_sigmoid.fit(X_train, y_train)

rf_for_isotonic = clone(random_forest)
rf_for_isotonic.fit(X_train, y_train)

sigmoid_calibrator = CalibratedClassifierCV(rf_for_sigmoid, method="sigmoid", cv="prefit")
isotonic_calibrator = CalibratedClassifierCV(rf_for_isotonic, method="isotonic", cv="prefit")

sigmoid_calibrator.fit(X_cal, y_cal)
isotonic_calibrator.fit(X_cal, y_cal)

model_probabilities = {
    "Logistic regression": raw_model_probabilities["Logistic regression"],
    "Random forest": raw_model_probabilities["Random forest"],
    "RF + sigmoid": sigmoid_calibrator.predict_proba(X_test)[:, 1],
    "RF + isotonic": isotonic_calibrator.predict_proba(X_test)[:, 1],
}

print("Calibration models fitted on the dedicated calibration split.")
        


## Compare the Calibrated Variants

Calibration should usually improve **Brier score** and **ECE** more directly than it improves ROC-AUC. That is the key distinction: calibration fixes confidence estimates, not necessarily ranking.
        


In [ ]:
comparison_metrics = pd.DataFrame(
    [summarize_probabilities(name, y_test, probs) for name, probs in model_probabilities.items()]
)
comparison_metrics["ece"] = comparison_metrics["model"].map(
    {
        name: expected_calibration_error(y_test, probs, n_bins=CONFIG["n_bins"])
        for name, probs in model_probabilities.items()
    }
)
comparison_metrics = comparison_metrics.sort_values(["brier_score", "ece", "roc_auc"])

display(comparison_metrics)
        


## Overlay the Reliability Curves

This single figure lets us compare raw and calibrated probability estimates directly.
        


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
color_map = {
    "Logistic regression": "#4C78A8",
    "Random forest": "#F58518",
    "RF + sigmoid": "#54A24B",
    "RF + isotonic": "#E45756",
}

for name, probs in model_probabilities.items():
    table = calibration_table(y_test, probs, n_bins=CONFIG["n_bins"])
    ax.plot(table["avg_confidence"], table["empirical_rate"], marker="o", linewidth=2, label=name, color=color_map[name])

ax.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive rate")
ax.set_title("Calibration Before and After Post-Processing")
ax.legend()
plt.show()
        


## What Did Calibration Actually Change?

A histogram before and after calibration shows whether the model became less overconfident or simply redistributed probability mass.
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
axes[0].hist(model_probabilities["Random forest"], bins=20, color="#F58518", edgecolor="white")
axes[0].set_title("Random forest probabilities")
axes[0].set_xlabel("Predicted probability")
axes[0].set_ylabel("Count")

axes[1].hist(model_probabilities["RF + isotonic"], bins=20, color="#E45756", edgecolor="white")
axes[1].set_title("Isotonic-calibrated probabilities")
axes[1].set_xlabel("Predicted probability")
plt.tight_layout()
plt.show()
        


## Pick a Model for Thresholding

Thresholding only makes sense once probabilities are meaningful. We'll choose the model with the best calibration quality on the test summary and treat its probabilities as the scores we want to convert into decisions.
        


In [ ]:
best_model_name = comparison_metrics.iloc[0]["model"]
best_probabilities = model_probabilities[best_model_name]

print(f"Using {best_model_name} for threshold analysis.")
print(f"Mean predicted positive probability: {best_probabilities.mean():.3f}")
        


## Build Thresholding Helpers

A threshold turns probabilities into class labels. We'll compute common decision metrics and also attach an **expected cost** that reflects our assumption that a false negative is six times as costly as a false positive.
        


In [ ]:
def threshold_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp)
    expected_cost = (CONFIG["fp_cost"] * fp + CONFIG["fn_cost"] * fn) / len(y_true)
    return {
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": fbeta_score(y_true, y_pred, beta=1, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "expected_cost": expected_cost,
    }


def threshold_sweep(y_true, y_prob, thresholds):
    return pd.DataFrame([threshold_metrics(y_true, y_prob, threshold) for threshold in thresholds])


def plot_confusion(ax, y_true, y_pred, title):
    matrix = confusion_matrix(y_true, y_pred)
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1], labels=[0, 1])
    ax.set_yticks([0, 1], labels=[0, 1])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            ax.text(col, row, matrix[row, col], ha="center", va="center", color="black")
    return image


## Start with the Default Threshold of 0.5

Many APIs default to 0.5, but that is only appropriate when the class prior, calibration, and error costs all support it.
        


In [ ]:
default_threshold_metrics = pd.DataFrame([threshold_metrics(y_test, best_probabilities, 0.5)])
display(default_threshold_metrics)
        


## Visualize How Threshold Changes the Confusion Matrix

Lowering the threshold catches more positives but also creates more false alarms. Raising it does the opposite.
        


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, threshold in zip(axes, [0.3, 0.5, 0.7]):
    predictions = (best_probabilities >= threshold).astype(int)
    plot_confusion(ax, y_test, predictions, title=f"Threshold = {threshold:.1f}")
plt.tight_layout()
plt.show()
        


## Plot the Precision-Recall and ROC Curves

These curves summarize how performance changes over all thresholds. For imbalanced problems, the precision-recall curve is often the more informative view.
        


In [ ]:
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, best_probabilities)
fpr_curve, tpr_curve, roc_thresholds = roc_curve(y_test, best_probabilities)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(recall_curve, precision_curve, color="#4C78A8", linewidth=2)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")

axes[1].plot(fpr_curve, tpr_curve, color="#E45756", linewidth=2)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()
        


## Sweep a Dense Grid of Thresholds

Now we'll evaluate a dense set of thresholds so we can see the trade-off curves directly and choose thresholds according to different objectives.
        


In [ ]:
threshold_grid = np.linspace(0.05, 0.95, 181)
threshold_results = threshold_sweep(y_test, best_probabilities, threshold_grid)

display(threshold_results.head())
        


## Precision, Recall, Specificity, and Cost Across Thresholds

This is the core thresholding picture: no single threshold is universally best. The right choice depends on which errors you care about most.
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(threshold_results["threshold"], threshold_results["precision"], label="precision", linewidth=2)
axes[0].plot(threshold_results["threshold"], threshold_results["recall"], label="recall", linewidth=2)
axes[0].plot(threshold_results["threshold"], threshold_results["specificity"], label="specificity", linewidth=2)
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Metric value")
axes[0].set_title("Decision Metrics by Threshold")
axes[0].legend()

axes[1].plot(threshold_results["threshold"], threshold_results["expected_cost"], color="#E45756", linewidth=2)
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Cost per example")
axes[1].set_title("Expected Cost by Threshold")

plt.tight_layout()
plt.show()
        


## Choose Thresholds for Different Operating Goals

We'll compare four threshold rules:

- default threshold of 0.50
- threshold that maximizes **F1**
- threshold that maximizes **F2** (recall-weighted)
- threshold that minimizes the expected business cost
        


In [ ]:
selected_rows = [
    ("Default 0.50", threshold_metrics(y_test, best_probabilities, 0.50)),
    ("Best F1", threshold_results.loc[threshold_results["f1"].idxmax()].to_dict()),
    ("Best F2", threshold_results.loc[threshold_results["f2"].idxmax()].to_dict()),
    ("Lowest expected cost", threshold_results.loc[threshold_results["expected_cost"].idxmin()].to_dict()),
]

selected_thresholds = pd.DataFrame(
    [{"rule": name, **metrics} for name, metrics in selected_rows]
).round(3)

display(selected_thresholds[["rule", "threshold", "precision", "recall", "f1", "f2", "expected_cost", "fp", "fn"]])
        


## Mark the Selected Thresholds on the Cost Curve

Seeing the selected rules on the same plot makes it easier to understand why they differ.
        


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(threshold_results["threshold"], threshold_results["expected_cost"], color="#E45756", linewidth=2)
ax.set_xlabel("Threshold")
ax.set_ylabel("Cost per example")
ax.set_title("Cost-Sensitive Threshold Selection")

palette = {
    "Default 0.50": "#4C78A8",
    "Best F1": "#54A24B",
    "Best F2": "#B279A2",
    "Lowest expected cost": "#F58518",
}

for _, row in selected_thresholds.iterrows():
    ax.axvline(row["threshold"], linestyle="--", color=palette[row["rule"]], alpha=0.8)
    ax.scatter(row["threshold"], row["expected_cost"], color=palette[row["rule"]], s=80, label=row["rule"])

ax.legend()
plt.show()
        


## Compare the Decision Outcomes Side by Side

The threshold rule changes the operational behavior of the same underlying model. Here we'll compare the confusion matrices for three representative choices.
        


In [ ]:
comparison_rules = selected_thresholds.set_index("rule").loc[["Default 0.50", "Best F2", "Lowest expected cost"]].reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (_, row) in zip(axes, comparison_rules.iterrows()):
    preds = (best_probabilities >= row["threshold"]).astype(int)
    title = f"{row['rule']} (threshold={row['threshold']:.2f})"
    plot_confusion(ax, y_test, preds, title=title)
plt.tight_layout()
plt.show()


## Reliability Diagrams and Thresholding Solve Different Problems

Calibration answers, "Can I trust 0.72 to mean roughly a 72% chance?" Thresholding answers, "At what probability should I take action?" They are complementary, not interchangeable.
        


In [ ]:
concept_summary = pd.DataFrame(
    [
        {
            "question": "How trustworthy are the probabilities?",
            "tool": "Calibration",
            "typical outputs": "Reliability diagram, Brier score, ECE, log loss",
        },
        {
            "question": "When should the model trigger a positive decision?",
            "tool": "Thresholding",
            "typical outputs": "Precision/recall trade-off, confusion matrix, cost curve",
        },
    ]
)

display(concept_summary)
        


## A Practical Checklist

In a real project, a good decision workflow usually looks like this:

1. Train a model and evaluate ranking quality.
2. Check whether its probabilities are calibrated.
3. Calibrate if needed, ideally on a dedicated validation split.
4. Choose a threshold that reflects the real cost of false positives and false negatives.
5. Revisit both calibration and threshold choice if the data distribution changes.
        


In [ ]:
final_checklist = pd.DataFrame(
    {
        "step": [1, 2, 3, 4, 5],
        "action": [
            "Measure discrimination with ROC-AUC / average precision.",
            "Inspect reliability diagrams and Brier score.",
            "Apply Platt scaling or isotonic regression if needed.",
            "Select a threshold using metrics or business cost.",
            "Monitor drift because calibration can decay over time.",
        ],
    }
)

display(final_checklist)
        


## Key Takeaways

- **Ranking quality** and **probability calibration** are different properties of a classifier.
- **Reliability diagrams**, **Brier score**, and **ECE** help diagnose whether predicted probabilities can be trusted.
- **Post-hoc calibration** can improve probability quality without meaningfully changing ranking metrics.
- A threshold of **0.5 is just a convention**, not a law.
- The best threshold depends on the operating goal, class imbalance, and the relative cost of errors.

This notebook fills the gap between model evaluation and decision-making. It is the bridge from "my model predicts scores" to "my system makes sensible choices." 
        
